In [1]:
from langchain_openai import OpenAI
from langchain_ollama import OllamaLLM
import prompts
import schemas
import json
from langchain_core.output_parsers import JsonOutputParser
import pandas as pd
from tqdm import tqdm
from langchain_core.prompts import PromptTemplate
import re
import glob
import os

/Users/hendrikweichel/miniconda3/envs/remedi/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in RunConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [2]:
with open("questions.json", "r") as f: 
    questions_json = json.load(f)
questions_json

[{'id': 'Business_Model_1',
  'question': "Does the following page contain a descriptions about the company's business activities.",
  'include': ['What the company produces or offers (products, services, or technologies)',
   'Its main business areas, divisions, or operating segments',
   'Core markets or regions of activity',
   'Key customers, suppliers, or industries it serves',
   'Business model, mission, or how it generates revenue'],
  'exclude': ['Governance or management reports',
   'Financial statements or numerical tables',
   'Sustainability KPIs, emissions data, or risk sections',
   'General introductions or letters to shareholders'],
  'prompt_template_id': 'baseline'}]

In [3]:
end_path = "data/results/"

In [4]:
reports_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/doc_search/JSONs"
reports_path = "JSONs"

In [5]:
report_paths = glob.glob(reports_path + "/*.json")
report_paths

['JSONs/Signify NV3.json',
 'JSONs/Nordnet AB1.json',
 'JSONs/Geberit AG1.json',
 'JSONs/Taylor Wimpey PLC1.json',
 'JSONs/TAG Immobilien AG3.json',
 'JSONs/Tenaris S.A.1.json',
 'JSONs/BKW AG1.json',
 'JSONs/Ashtead Group plc1.json',
 'JSONs/Anglo American plc1.json',
 'JSONs/RWE AG1.json',
 'JSONs/Enel SpA1.json',
 'JSONs/Bavarian Nordic AS1.json',
 'JSONs/Scout24 SE3.json',
 'JSONs/Publicis Groupe SA1.json',
 'JSONs/UniCredit S.p.A.3.json',
 'JSONs/DiaSorin S.p.A.1.json',
 'JSONs/LondonMetric Property PLC1.json',
 'JSONs/Orkla ASA1.json',
 'JSONs/Sulzer AG2.json',
 'JSONs/CD Projekt S.A.1.json',
 'JSONs/Prysmian S.p.A.1.json',
 'JSONs/Grafton Group Plc1.json',
 'JSONs/EQT AB1.json',
 'JSONs/Accelleron Industries AG1.json',
 'JSONs/shell-annual-report-2022.json',
 'JSONs/DCC Plc1.json',
 'JSONs/Wendel SE2.json',
 'JSONs/ASM International N.V.1.json',
 'JSONs/Vonovia SE1.json',
 'JSONs/Pernod Ricard SA2.json',
 'JSONs/Segro PLC1.json',
 'JSONs/Intermediate Capital Group plc3.json',
 '

In [6]:
# Function to get a language model object (Ollama or ChatGPT)
def get_language_model(model_type: str = "ollama", model_name: str = "default-model"):
    """
    Returns a language model object (Ollama or ChatGPT).
    
    Args:
        model_type (str): The type of model to use ("ollama" or "chatgpt"). Default is "ollama".
        model_name (str): The name of the model to use. Default is "default-model".
    
    Returns:
        An instance of the selected language model.
    """
    if model_type.lower() == "ollama":
        return OllamaLLM(model=model_name)
    elif model_type.lower() == "chatgpt":
        return OpenAI(model=model_name)
    else:
        raise ValueError("Invalid model_type. Choose 'ollama' or 'chatgpt'.")

# Function to interact with thea language model
def query_llm(llm, question: str, page: str):
    """
    Queries the language model with a question and a page string.
    
    Args:
        llm: The language model object.
        question (str): The question to ask the model.
        page (str): The page string to provide context.
    
    Returns:
        str: The output of the language model.
    """
    prompt = f"Context: {page}\n\nQuestion: {question}"
    return llm.invoke(prompt)

In [17]:
llm = get_language_model(model_name="llama3:8b")
parser = JsonOutputParser()

In [18]:
# results = []

# for page in tqdm(report["pages"]): 
#     prompt = prompts.render_prompt(
#             questions_[0],
#             page["markdown"],
#             prompt_template_id="baseline",
#             few_shot=False,
#             truncate_at=None,
#         )

#     llm_response = llm.invoke(prompt.text)

#     match = re.search(r"\{.*\}", llm_response, re.DOTALL)
#     if not match:
#         print(llm_response)
#         #raise ValueError("No JSON found in LLM output!")
    

#     json_str = match.group(0)

#     parsed = parser.parse(json_str)

#     result_page = parsed
#     result_page["page"] = page["page"]
#     result_page["markdown"] = page["markdown"]
#     results.append(result_page)

# df = pd.DataFrame(results)

In [19]:
questions = [schemas.QuestionSpec(**q) for q in questions_json]
questions

[QuestionSpec(id='Business_Model_1', question="Does the following page contain a descriptions about the company's business activities.", include=['What the company produces or offers (products, services, or technologies)', 'Its main business areas, divisions, or operating segments', 'Core markets or regions of activity', 'Key customers, suppliers, or industries it serves', 'Business model, mission, or how it generates revenue'], exclude=['Governance or management reports', 'Financial statements or numerical tables', 'Sustainability KPIs, emissions data, or risk sections', 'General introductions or letters to shareholders'], prompt_template_id='baseline', few_shot_examples=[])]

In [20]:
from langchain_core.exceptions import OutputParserException

In [21]:
for report_path in tqdm(report_paths[19:]): 

    with open(report_path, "r") as f: 
        report = json.load(f)

    report_result_path = os.path.join(end_path, os.path.basename(report_path))
    os.makedirs(report_result_path, exist_ok =True)

    for question in questions:
        results = []
        for page in tqdm(report["pages"][:50]): 
            prompt = prompts.render_prompt(
                    question,
                    page["markdown"],
                    prompt_template_id="baseline",
                    few_shot=False,
                    truncate_at=None,
                )

            llm_response = llm.invoke(prompt.text)

            match = re.search(r"\{.*\}", llm_response, re.DOTALL)
            if match:
                json_str = match.group(0)

                try: 
                    parsed = parser.parse(json_str)

                    result_page = parsed
                    result_page["page"] = page["page"]
                    result_page["markdown"] = page["markdown"]
                    results.append(result_page)

                except OutputParserException: 
                    continue

        df = pd.DataFrame(results)
        df = df.sort_values(by=["decision", "page", "confidence"], ascending=[False, True, False])
        df.to_csv(os.path.join(report_result_path, f"{question.id}.csv"))

        positive_pages = df[df["decision"] == "yes"]
        positive_text = ""
        for i, page in positive_pages.iterrows(): 
            positive_text += f'Nbr {i}, Page {page["page"]}\n' + 40 * '-' + f'\n\n{page["markdown"]}\n\n'
        
        with open(os.path.join(report_result_path, f"{question.id}_positives.txt"), "w") as f: 
            f.write(positive_text)

  6%|▌         | 14/242 [1:32:04<24:59:27, 394.59s/it]


KeyboardInterrupt: 

In [ ]:
a = parser.parse(json_str)
a

In [ ]:
print(json_str)

{
  "decision": "yes",
  "confidence": 0.8,
  "spans": [
    {
      "text": "\\Financial assets\\The Group classifies each financial asset upon initial recognition into of one of four categories of financial assets, which are distinguished based on the Group's business model for managing the assets and the characteristics of the contractual cash flows:\\. .\\. \
- \\(1) assets measured at amortized cost after initial recognition;\\. .\\. \
- \\(2) assets measured at fair value through other comprehensive income after initial recognition;\\. .\\. \
- \\(3) assets measured at fair value through profit or loss;\\. .\\. \
- \\(4) hedging financial instruments.\\ .\\. \
The classification of financial assets is made upon initial recognition and can only be changed if the business model for managing financial assets changes\\. The principal models for managing financial assets include the model of holding for receiving contractual cash flows, the model of holding for receiving contractual c

In [ ]:
df = pd.DataFrame(results)
df

,decision,confidence,spans,rationale,page,markdown
0,no,0.0,[],No information provided about a transition pla...,1,
1,no,0.5,[{'text': ''}],The provided page content appears to be a fina...,2,<!-- image -->\n\n## CONSOLIDATED FINANCIAL ST...
2,no,0.0,[{'text': ''}],The provided content does not address the ques...,3,Disclaimer\n\nThis English language translatio...
3,no,0.8,[{'text': ''}],The provided content does not describe the com...,4,<!-- image -->\n\n## CD PROJEKT Group - Select...
4,no,0.8,[{'text': ''}],The provided page content does not contain any...,5,<!-- image -->\n\n## Table of contents\n\n| Re...
5,no,0.0,[],There is no information provided on a credible...,6,<!-- image -->\n\nNote 2. Operating expenses ...
6,no,0.5,[{'text': ''}],The page content does not mention any transiti...,7,<!-- image -->\n\n## Key financial data of the...
7,no,0.8,[{'text': ''}],The provided content does not describe a credi...,8,<!-- image -->\n\n## Consolidated income state...
8,no,0.8,[{'text': 'There is no information about a cre...,The provided content only focuses on financial...,9,<!-- image -->\n\n## Consolidated statement of...
9,no,0.5,[{'text': ''}],The provided content appears to be a balance s...,10,<!-- image -->\n\n| ...


In [ ]:
print(positive_text)

Nbr 37, Page 38
----------------------------------------

<!-- image -->

## Operating segments

## Presentation of the financial statements taking into account operating segments

The scope of the financial information provided on the Group's operating segments is consistent with the requirements of IFRS 8. The segments' results are determined based on their net profits.

## Description of differences in the basis for the determination of segments and the profit or loss of a segment compared with the last annual consolidated financial statements

The Group did not make any changes in determining segments or in the measurement of the profits or losses of the individual segments in relation to the financial statements for the year ended 31 December 2021.

There are no differences between the measurement of the assets, liabilities, profits and losses of the Group's reporting segments.

## Operating segments

In 2022, the Group's operations were carried out in two business segments:

-  